# Estimate Costs - TikToken

Tiktoken is the official OpenAI's tokenizer library for python. It can tokenize text using a specific encoding and allows you to count how many tokens a string contains.

Embedding models' usage is billed by input tokens:

| model | cost (1 page = ~800 tokens) | cost per token | max input | tokenizer encoding |
| ---- | --- | --- | --- | --- |
| OpenAI text-embedding-3-small | 62,500 pages per /$ | 0,02$ per 1kk tokens | 8192 tokens | cl100k_base |
| OpenAI text-embedding-3-large | 9,615 pages per /$  | 0,13$ per 1kk tokens | 8192 tokens | cl100k_base |


This is important because you can __predict the cost to index your dataset for a RAG__ (create embeddings to store in a DB) before actually doing the embedding:
- Use tiktoken to compute tokens of your dataset
- Use the number of tokens to compute total embedding cost

Notes: 
- Encodings specify how text is transformed in tokens (they have a vocabulary of tokens -> ids)
- Each model use an encoding and so you must provide the right encoding name to tiktoken to predict costs of a specific model.
- You cannot use tiktoken to predict the cost of an LLM interaction, because there also output tokens are involved in the costing, in that case you must see the real usage.

## Install

```bash 
    poetry add tiktoken
```

## Usage

In [3]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

res = num_tokens_from_string("tiktoken is great!", "cl100k_base")
print(f"number of tokens: {res}")

number of tokens: 6


## Details

Show encodings for various models

In [9]:
import tiktoken

print(f"encoding for text-embedding-3-small: {tiktoken.encoding_for_model("text-embedding-3-small")}")
print(f"encoding for text-embedding-3-large: {tiktoken.encoding_for_model("text-embedding-3-large")}")
print(f"encoding for gpt-5-nano: {tiktoken.encoding_for_model("gpt-5-nano")}")
print(f"encoding for gpt-5.6-sol: {tiktoken.encoding_for_model("gpt-5.6-sol")}")

print("")
print(f"encoding name for text-embedding-3-small: {tiktoken.encoding_name_for_model("text-embedding-3-small")}")
print(f"encoding name for text-embedding-3-large: {tiktoken.encoding_name_for_model("text-embedding-3-large")}")
print(f"encoding name for gpt-5-nano: {tiktoken.encoding_name_for_model("gpt-5-nano")}")
print(f"encoding name for gpt-5.6-sol: {tiktoken.encoding_name_for_model("gpt-5.6-sol")}")

encoding for text-embedding-3-small: <Encoding 'cl100k_base'>
encoding for text-embedding-3-large: <Encoding 'cl100k_base'>
encoding for gpt-5-nano: <Encoding 'o200k_base'>
encoding for gpt-5.6-sol: <Encoding 'o200k_base'>

encoding name for text-embedding-3-small: cl100k_base
encoding name for text-embedding-3-large: cl100k_base
encoding name for gpt-5-nano: o200k_base
encoding name for gpt-5.6-sol: o200k_base


Tokenize a text and show tokens

In [32]:
import tiktoken

text = "this is a sample text to tokenize wowwwwww"

# get the encoding
encoding = tiktoken.get_encoding("cl100k_base")

# encode text (tokenize) and show tokens 
tokens_ids = encoding.encode(text)
tokens = [encoding.decode_single_token_bytes(t).decode() for t in tokens_ids] 

# print
print(f"tokens' ids: {tokens_ids}")
print(f"tokens: {tokens}")
print(f"num of tokens: {len(tokens_ids)}")

tokens' ids: [576, 374, 264, 6205, 1495, 311, 78751, 36765, 1273, 2185]
tokens: ['this', ' is', ' a', ' sample', ' text', ' to', ' tokenize', ' wow', 'ww', 'www']
num of tokens: 10


Verify predicition

In [33]:
from openai import OpenAI

client = OpenAI()

result = client.embeddings.create(input=text,model="text-embedding-3-small")

print(f"usage: {result.usage}")
print(f"embeddings: {result.data[0]}")
print(f"cost: {result.usage.total_tokens/1000000*0.02:.8f}$")

usage: Usage(prompt_tokens=10, total_tokens=10)
embeddings: Embedding(embedding=[0.0182342529296875, -0.007495880126953125, 0.0229034423828125, 0.033599853515625, -0.0174407958984375, -0.062225341796875, -0.0168609619140625, -0.0128021240234375, -0.016754150390625, 0.0188751220703125, 0.0257720947265625, 0.003780364990234375, -0.01194000244140625, -0.0240631103515625, 0.036712646484375, 0.04022216796875, 0.0014972686767578125, 0.0244903564453125, 0.004238128662109375, 0.0292510986328125, -0.0017604827880859375, -0.01210784912109375, -0.04046630859375, 0.01085662841796875, 0.0011768341064453125, -0.00461578369140625, -0.0179595947265625, 0.04046630859375, 0.07122802734375, -0.04144287109375, -0.0221710205078125, -0.04327392578125, 0.004146575927734375, -0.0531005859375, 0.00626373291015625, 0.01432037353515625, -0.0265655517578125, 0.013824462890625, 0.005237579345703125, -0.036865234375, 0.00363922119140625, -0.057220458984375, 0.0005679130554199219, 0.05291748046875, -0.00110054016113